# Prod Baseline Evaluation

> **Status:** Exploratory — superseded by `notebooks/production/train_model_prod.ipynb`.
> This notebook was used to establish the initial baseline from the first production data pull. It captures the early model comparison work that led to XGBoost being selected. Do not use it to regenerate model artifacts.

**Goal:** Get a real baseline from the prod invoice pull.

**Plan:**
1. Join invoices to payment rows
2. Build `paid_within_30_days`
3. Check class balance
4. Set dumb baselines
5. See what logistic regression can do

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, log_loss, precision_score, recall_score, roc_auc_score, average_precision_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

## Step 2: Load and Join Datasets

Join `invoice_id` to `invoiceid`.

For each invoice, take the first successful payment and ask: did it happen within 30 days?

- `paid_within_30_days = 1`: yes
- `paid_within_30_days = 0`: no

In [ ]:
DATA_DIR = "../../data/raw"

invoices = pd.read_csv(f"{DATA_DIR}/prod_invoices_dataset.csv")
payments = pd.read_csv(f"{DATA_DIR}/prod_invoice_payment_rows.csv")

invoices["createdat"] = pd.to_datetime(invoices["createdat"], utc=True, errors="coerce")
payments["submittedat"] = pd.to_datetime(
    payments["submittedat"],
    utc=True,
    errors="coerce",
    format="mixed",
)

print(f"Invoices: {len(invoices):,}")
print(f"Payment rows: {len(payments):,}")
print(f"createdat parse nulls: {invoices['createdat'].isna().sum():,}")
print(f"submittedat parse nulls: {payments['submittedat'].isna().sum():,}")

In [ ]:
earliest_success = (
    payments.loc[payments["paymentstatus"] == 1, ["invoiceid", "submittedat"]]
    .groupby("invoiceid", as_index=False)["submittedat"]
    .min()
    .rename(columns={"submittedat": "first_succeeded_at"})
)

df = invoices.merge(
    earliest_success,
    left_on="invoice_id",
    right_on="invoiceid",
    how="left",
)

df["days_to_payment"] = (df["first_succeeded_at"] - df["createdat"]).dt.total_seconds() / 86400
df["paid_within_30_days"] = df["days_to_payment"].le(30).fillna(False).astype(int)

print(f"Labeled rows: {len(df):,}")
print(df["paid_within_30_days"].value_counts().rename({0: "not paid", 1: "paid"}))

## Step 3: Sanity Check the Joined Data

In [ ]:
counts = df["paid_within_30_days"].value_counts().sort_index()

print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nClass balance:\n", counts)
print(f"\nPaid rate: {counts.get(1, 0) / len(df):.1%}")

In [ ]:
# Feature distributions vs label
feature_cols = ["amount", "account_age_at_invoice_time", "has_payment_method_on_file",
                "number_of_payment_methods", "has_card_on_file", "has_bank_on_file"]

df[feature_cols + ["paid_within_30_days"]].groupby("paid_within_30_days").mean().T

## Step 4: Easy Baselines

The real model has to beat these.

- Majority class: predict everyone pays
- Prior probability: predict the overall pay rate for everyone

In [ ]:
y = df["paid_within_30_days"].to_numpy()
paid_rate = y.mean()

majority_metrics = classification_metrics(y, np.ones(len(y), dtype=int))
prior_probs = np.full(len(y), paid_rate)

print("=== Majority Class Baseline ===")
for name, value in majority_metrics.items():
    print(f"{name.title():<10} {value:.3f}")

print("\n=== Prior Probability Baseline ===")
print(f"Paid rate   {paid_rate:.3f}")
print(f"Brier       {brier_score_loss(y, prior_probs):.4f}")
print(f"Log-loss    {log_loss(y, prior_probs):.4f}")

## Step 5: Check Features One at a Time

Quick read on whether any single feature already has signal.

In [ ]:
results = []

for col in feature_cols:
    values = pd.to_numeric(df[col], errors="coerce").fillna(0)
    preds = values.astype(int) if values.nunique() <= 2 else values.ge(values.median()).astype(int)
    results.append({"feature": col, **classification_metrics(y, preds)})

pd.DataFrame(results).sort_values("f1", ascending=False).round(3)

## Step 6: Logistic Regression Baseline

This is the first real baseline.

If a more complex model can't beat it cleanly, that's a warning sign.

In [ ]:
X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
)

auc_scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc")
f1_scores = cross_val_score(model, X, y, cv=5, scoring="f1")

print("=== Logistic Regression Baseline (5-fold CV) ===")
print(f"AUC-ROC  {auc_scores.mean():.3f} +/- {auc_scores.std():.3f}")
print(f"F1       {f1_scores.mean():.3f} +/- {f1_scores.std():.3f}")
print("\nFeatures:", feature_cols)

## Step 6b: Defensive Layer 1 — Precision-Recall Curve & Threshold Tuning

The default threshold of 0.5 assumes equal costs for false positives and false negatives.

For imbalanced data, this is almost always wrong. Tune the threshold by finding the operating point on the PR curve that matches your business metric.

Example: "I want to catch 70% of non-payers (recall = 0.7), what's my precision?"

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

# Compute PR curve out-of-fold to find optimal threshold
all_y_true, all_y_pred_proba = [], []
for train_idx, test_idx in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y[train_idx], y[test_idx]
    model.fit(Xtr, ytr)
    all_y_true.extend(yte)
    all_y_pred_proba.extend(model.predict_proba(Xte)[:, 1])

all_y_true = np.array(all_y_true)
all_y_pred_proba = np.array(all_y_pred_proba)

# Find threshold that gives 70% recall on paid_within_30_days (non-payers in negative class)
precisions, recalls, thresholds = precision_recall_curve(all_y_true, all_y_pred_proba)

# Find threshold closest to recall=0.7 (catch 70% of payers at some precision level)
idx_70 = np.argmin(np.abs(recalls - 0.7))
threshold_70_recall = thresholds[idx_70] if idx_70 < len(thresholds) else 0.5
precision_at_70 = precisions[idx_70]

print("=== Precision-Recall Threshold Tuning ===")
print(f"At 70% recall: threshold={threshold_70_recall:.3f}, precision={precision_at_70:.3f}")
print(f"Default (threshold=0.5) recall: {recalls[np.argmin(np.abs(thresholds - 0.5))]:.3f}")
print("\nPR curve gives you flexibility:")
print("  - High precision, low recall: catch fewer non-payers but confident")
print("  - Low precision, high recall: catch more non-payers but more false alarms")

## Step 6c: Defensive Layer 2 — SMOTE Resampling

Synthetic Minority Oversampling generates synthetic non-payer examples to balance the training data.

This is the industry standard for imbalanced classification. Run SMOTE only on the training fold (never on test data).

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold

# Evaluate logistic regression WITH SMOTE on training data
smote_aucs, smote_f1s, smote_precisions = [], [], []

for train_idx, test_idx in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y):
    Xtr, Xte = X.iloc[train_idx].values, X.iloc[test_idx].values
    ytr, yte = y[train_idx], y[test_idx]
    
    # Apply SMOTE only to training data
    smote = SMOTE(random_state=42)
    Xtr_resampled, ytr_resampled = smote.fit_resample(Xtr, ytr)
    
    # Train on resampled data, evaluate on original test data
    model_smote = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    model_smote.fit(Xtr_resampled, ytr_resampled)
    proba = model_smote.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    
    smote_aucs.append(roc_auc_score(yte, proba))
    smote_f1s.append(f1_score(yte, pred, zero_division=0))
    smote_precisions.append(precision_score(yte, pred, zero_division=0))

print("=== Logistic Regression WITH SMOTE (5-fold CV) ===")
print(f"AUC-ROC  {np.mean(smote_aucs):.3f} +/- {np.std(smote_aucs):.3f} (baseline: {auc_scores.mean():.3f})")
print(f"F1       {np.mean(smote_f1s):.3f} +/- {np.std(smote_f1s):.3f} (baseline: {f1_scores.mean():.3f})")
print(f"Precision {np.mean(smote_precisions):.3f}")
print("\nSMOTE helps balance the training data to learn minority class patterns better.")

## Step 6d: Defensive Layer 3 — Lift & Concentration Metrics

*This is the fraud-detection industry standard.* 

Rank predictions by score and ask: "What fraction of actual non-payers are in my top-K% of risk?"

A good model captures 70-80% of non-payers in the top 20% of scores. A random model captures 20%.

In [ ]:
def compute_lift_metrics(y_true, y_pred_proba, percentiles=[10, 20, 30]):
    """
    Given predictions ranked by score, what % of positive class is captured 
    in top K% of scores?
    
    Lift = (captured % / base %) 
    Example: If 60% of minority class is in top 20%, lift = 60/20 = 3.0x
    """
    df_scores = pd.DataFrame({
        'y_true': y_true,
        'score': y_pred_proba
    })
    df_scores['rank_pct'] = pd.qcut(df_scores['score'], q=100, labels=False, duplicates='drop') / 100
    
    results = {}
    for pct in percentiles:
        top_k = df_scores[df_scores['rank_pct'] >= (1 - pct / 100)]
        captured_minority = top_k[top_k['y_true'] == 1].shape[0]
        total_minority = (df_scores['y_true'] == 1).sum()
        base_rate = pct / 100
        
        captured_pct = captured_minority / total_minority if total_minority > 0 else 0
        lift = captured_pct / base_rate if base_rate > 0 else 1.0
        
        results[f'top_{pct}pct'] = {
            'captured_minority': captured_minority,
            'captured_pct': captured_pct,
            'lift': lift
        }
    
    return results

# Compute lift on out-of-fold predictions
lift = compute_lift_metrics(all_y_true, all_y_pred_proba, percentiles=[10, 20, 30])

print("=== Lift Metrics (Out-of-Fold) ===")
print("(% of minority class captured in top K% of risk scores)")
print()
for key, metrics in lift.items():
    print(f"{key}:")
    print(f"  Captured: {metrics['captured_pct']:.1%} of non-payers")
    print(f"  Lift: {metrics['lift']:.2f}x (1.0x = random, >1.5x = good)")
    print()

print("Random model would capture ~10% in top 10%, ~20% in top 20%, etc.")
print("A good model should have lift > 1.5 in top deciles.")

## Step 7: Protect Against Imbalance — Defensive Summary

Four defensive layers ensure your model doesn't just "predict everyone pays":

| Layer | Technique | Why | Implementation |
|---|---|---|---|
| **1. Metric Choice** | Use AUC-ROC, PR-AUC, F1 (not accuracy) | Accuracy is useless on 80% base rate | ✓ Done |
| **2. Stratified CV** | StratifiedKFold, not random splits | Ensures minority class in every fold | ✓ Done |
| **3. Class Weighting** | `class_weight='balanced'` in LogisticRegression | Penalize model for missing non-payers | ✓ Done |
| **4. Threshold Tuning** | Use PR curve, don't use 0.5 | Match business cost of false positives vs false negatives | ✓ Added (Step 6b) |
| **5. SMOTE Resampling** | Synthetic oversampling of minority | More training examples for rare class | ✓ Added (Step 6c) |
| **6. Lift/Concentration** | What % of non-payers in top K%? | Fraud-detection standard; directly actionable | ✓ Added (Step 6d) |

**Pass/fail gate:** Model must achieve >1.5x lift in top 20% (i.e., capture >30% of non-payers with 20% of predictions).

## Step 8: Train Candidate Models — Random Forest & XGBoost

Now test whether more complex models can beat the lift threshold (1.5x in top 20%).

Both models trained with SMOTE resampling, evaluated on same StratifiedKFold split for fair comparison.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store all predictions for lift computation
rf_all_y_true, rf_all_y_pred_proba = [], []
xgb_all_y_true, xgb_all_y_pred_proba = [], []

rf_aucs, rf_f1s, rf_lifts_top20 = [], [], []
xgb_aucs, xgb_f1s, xgb_lifts_top20 = [], [], []

for train_idx, test_idx in cv.split(X, y):
    Xtr, Xte = X.iloc[train_idx].values, X.iloc[test_idx].values
    ytr, yte = y[train_idx], y[test_idx]
    
    # Apply SMOTE only to training data
    smote = SMOTE(random_state=42, k_neighbors=3)
    Xtr_resampled, ytr_resampled = smote.fit_resample(Xtr, ytr)
    
    # Random Forest
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
    rf_model.fit(Xtr_resampled, ytr_resampled)
    rf_proba = rf_model.predict_proba(Xte)[:, 1]
    rf_pred = (rf_proba >= 0.5).astype(int)
    rf_aucs.append(roc_auc_score(yte, rf_proba))
    rf_f1s.append(f1_score(yte, rf_pred, zero_division=0))
    
    rf_all_y_true.extend(yte)
    rf_all_y_pred_proba.extend(rf_proba)
    
    # XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=(ytr_resampled == 0).sum() / (ytr_resampled == 1).sum(),
        random_state=42,
        n_jobs=-1,
        eval_metric='auc'
    )
    xgb_model.fit(Xtr_resampled, ytr_resampled, verbose=False)
    xgb_proba = xgb_model.predict_proba(Xte)[:, 1]
    xgb_pred = (xgb_proba >= 0.5).astype(int)
    xgb_aucs.append(roc_auc_score(yte, xgb_proba))
    xgb_f1s.append(f1_score(yte, xgb_pred, zero_division=0))
    
    xgb_all_y_true.extend(yte)
    xgb_all_y_pred_proba.extend(xgb_proba)

rf_all_y_true = np.array(rf_all_y_true)
rf_all_y_pred_proba = np.array(rf_all_y_pred_proba)
xgb_all_y_true = np.array(xgb_all_y_true)
xgb_all_y_pred_proba = np.array(xgb_all_y_pred_proba)

print("=== Random Forest with SMOTE (5-fold CV) ===")
print(f"AUC-ROC  {np.mean(rf_aucs):.3f} +/- {np.std(rf_aucs):.3f}")
print(f"F1       {np.mean(rf_f1s):.3f} +/- {np.std(rf_f1s):.3f}")

print("\n=== XGBoost with SMOTE (5-fold CV) ===")
print(f"AUC-ROC  {np.mean(xgb_aucs):.3f} +/- {np.std(xgb_aucs):.3f}")
print(f"F1       {np.mean(xgb_f1s):.3f} +/- {np.std(xgb_f1s):.3f}")

In [ ]:
# Compute lift metrics for all three models
rf_lift = compute_lift_metrics(rf_all_y_true, rf_all_y_pred_proba, percentiles=[10, 20, 30])
xgb_lift = compute_lift_metrics(xgb_all_y_true, xgb_all_y_pred_proba, percentiles=[10, 20, 30])

# Build comparison table
comparison = {
    "Logistic Regression": {
        "AUC": np.mean(auc_scores),
        "F1": np.mean(f1_scores),
        "Lift@20%": lift["top_20pct"]["lift"],
        "Captured@20%": lift["top_20pct"]["captured_pct"]
    },
    "Random Forest": {
        "AUC": np.mean(rf_aucs),
        "F1": np.mean(rf_f1s),
        "Lift@20%": rf_lift["top_20pct"]["lift"],
        "Captured@20%": rf_lift["top_20pct"]["captured_pct"]
    },
    "XGBoost": {
        "AUC": np.mean(xgb_aucs),
        "F1": np.mean(xgb_f1s),
        "Lift@20%": xgb_lift["top_20pct"]["lift"],
        "Captured@20%": xgb_lift["top_20pct"]["captured_pct"]
    }
}

comparison_df = pd.DataFrame(comparison).T

print("\n" + "="*70)
print("MODEL COMPARISON — ALL DEFENSES APPLIED (SMOTE + StratifiedKFold)")
print("="*70)
print(comparison_df.round(3))
print("\n✓ PASS CRITERIA:")
print("  - AUC improvement over logistic: >= 0.02")
print("  - Lift@20% threshold: >= 1.5x (to qualify for production)")
print("  - Captured@20%: >= 30% of non-payers")

best_lift = max(comparison["Random Forest"]["Lift@20%"], comparison["XGBoost"]["Lift@20%"])
if best_lift >= 1.5:
    print(f"\n✓ MODEL IS VIABLE: Best lift = {best_lift:.2f}x (exceeds 1.5x threshold)")
    winner = "XGBoost" if comparison["XGBoost"]["Lift@20%"] > comparison["Random Forest"]["Lift@20%"] else "Random Forest"
    print(f"  → Recommend {winner} for deployment")
else:
    print(f"\n✗ MODEL NEEDS WORK: Best lift = {best_lift:.2f}x (below 1.5x threshold)")
    print("  → Need feature engineering, ensemble stacking, or cost-sensitive tuning")

## Step 9: Diagnosis & Next Steps

The model is not viable with current features. Lift is only ~1.14x (threshold: 1.5x).

**What this means:** 
- Using amount, account_age, payment_method flags gives almost no separation
- A random model would catch 20% of non-payers in top 20% of scores
- Your model catches only 23%—barely better

**Root cause:** These features are *account-level* but non-payment is *behavioral* (invoice-specific or temporal).

**To reach viability, you need:**

In [ ]:
print("FEATURE IMPORTANCE (from XGBoost):")
print("="*60)

# Train XGBoost one more time on full data to get feature importance
xgb_full = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='auc')
xgb_full.fit(X, y, verbose=False)

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_full.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.to_string(index=False))

print("\n" + "="*60)
print("WHAT'S MISSING (features that would improve lift):")
print("="*60)
print("""
1. PAYMENT HISTORY (highest priority)
   - Days since last successful payment (recency)
   - Number of late payments in past 6 months
   - Average days to payment (historical conversion speed)
   
2. INVOICE CHARACTERISTICS
   - Days since invoice created (older invoices ≈ higher risk)
   - Invoice amount vs account's typical invoice size (outlier risk)
   - Number of open invoices (cash flow stress)
   
3. ACCOUNT BEHAVIORAL SIGNALS
   - Last login date / activity recency
   - Communication engagement (emails opened, links clicked)
   - Account health score (internal metric)
   
4. TEMPORAL / EXTERNAL
   - Seasonality (month of year, fiscal calendar alignment)
   - Days of week (Fridays vs Mondays for payment)
   - Macro signals (economy, industry defaults)
   
WHY: Non-payment is predictable from *what the account did before*.
     The current features are static account properties (amount, age).
     You need DYNAMIC features (behavior, history, trends).
""")

print("="*60)
print("RECOMMENDED ACTION PLAN:")
print("="*60)
print("""
1. ✓ CURRENT STATE: Proved baseline is <1.5x lift (not viable)
2. NEXT: Engineer payment history features (30m, high ROI)
   - Query payments table for historical patterns
   - Compute days_to_payment for each invoice in last N months
   - Aggregate into: median_days, pct_late, recent_failures
3. TEST: Retrain with history features (15m)
   - If lift improves to >1.5x, model is viable
   - If not, add behavioral signals
4. VALIDATE: Deploy on holdout test set with business metrics
   - Rank all open invoices by risk
   - Measure lift on real collections outcomes
5. ITERATE: Monitor non-payer distribution in top 20% decile
""")

## Step 10: Feature Engineering — Payment History

Add behavioral features derived from payment history.

Goal: Engineer features that capture *what the account did before*, not just static properties.

In [ ]:
# Step 10a: Compute account-level payment history aggregates
print("Computing account-level payment history features...")

# Get all successful payments and parse dates
account_payments = payments[payments["paymentstatus"] == 1].copy()
account_payments["submittedat"] = pd.to_datetime(account_payments["submittedat"], utc=True, errors="coerce", format="mixed")

# Prepare invoice data
invoice_subset = invoices[["invoice_id", "person_id_obfuscated", "createdat"]].copy()
invoice_subset.rename(columns={"createdat": "invoice_createdat"}, inplace=True)

# Join payments to invoices with explicit suffix to avoid conflicts
payment_with_invoice = account_payments.merge(
    invoice_subset,
    left_on="invoiceid",
    right_on="invoice_id",
    how="left",
    suffixes=("_payment", "_invoice")
)

# Compute days to payment (from invoice creation to payment submission)
payment_with_invoice["days_to_payment_historical"] = (
    payment_with_invoice["submittedat"] - payment_with_invoice["invoice_createdat"]
).dt.total_seconds() / 86400

# Compute account-level aggregates (grouped by person_id)
account_stats = payment_with_invoice.groupby("person_id_obfuscated").agg({
    "days_to_payment_historical": ["median", lambda x: (x > 30).mean()],
    "invoiceid": "count",
    "submittedat": "max"
}).reset_index()

account_stats.columns = ["person_id_obfuscated", "median_days_to_payment", "pct_late_payments", "num_successful_payments", "last_payment_date"]

# Join account stats back to main dataset
df_enhanced = df.copy()
df_enhanced = df_enhanced.merge(
    account_stats,
    on="person_id_obfuscated",
    how="left"
)

# Compute days since last payment relative to invoice creation time
df_enhanced["days_since_last_payment"] = (
    df_enhanced["createdat"] - df_enhanced["last_payment_date"]
).dt.total_seconds() / 86400

# Days since invoice created (temporal feature)
df_enhanced["days_since_invoice_created"] = (
    pd.Timestamp.now(tz='UTC') - df_enhanced["createdat"]
).dt.total_seconds() / 86400

# Fill nulls with conservative defaults (new accounts, no history)
df_enhanced["median_days_to_payment"] = df_enhanced["median_days_to_payment"].fillna(30)
df_enhanced["pct_late_payments"] = df_enhanced["pct_late_payments"].fillna(0)
df_enhanced["num_successful_payments"] = df_enhanced["num_successful_payments"].fillna(0)
df_enhanced["days_since_last_payment"] = df_enhanced["days_since_last_payment"].fillna(365)

print(f"Account stats computed: {len(account_stats)} accounts")
print(f"\nNew features added:")
print(f"  - median_days_to_payment: {df_enhanced['median_days_to_payment'].describe().round(1)}")
print(f"  - pct_late_payments: {df_enhanced['pct_late_payments'].describe().round(3)}")
print(f"  - num_successful_payments: {df_enhanced['num_successful_payments'].describe().round(1)}")
print(f"  - days_since_last_payment: {df_enhanced['days_since_last_payment'].describe().round(1)}")
print(f"  - days_since_invoice_created: {df_enhanced['days_since_invoice_created'].describe().round(1)}")

In [ ]:
# Step 10b: Retrain models with expanded feature set
print("\n" + "="*70)
print("RETRAINING WITH PAYMENT HISTORY FEATURES")
print("="*70)

# New feature set includes payment history
feature_cols_expanded = feature_cols + [
    "median_days_to_payment",
    "pct_late_payments",
    "num_successful_payments",
    "days_since_last_payment",
    "days_since_invoice_created"
]

X_expanded = df_enhanced[feature_cols_expanded].apply(pd.to_numeric, errors="coerce").fillna(0)
y_expanded = df_enhanced["paid_within_30_days"].to_numpy()

# Evaluate all three models with expanded features
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_all_y_true_exp, lr_all_y_pred_proba_exp = [], []
rf_all_y_true_exp, rf_all_y_pred_proba_exp = [], []
xgb_all_y_true_exp, xgb_all_y_pred_proba_exp = [], []

lr_aucs_exp, rf_aucs_exp, xgb_aucs_exp = [], [], []
lr_f1s_exp, rf_f1s_exp, xgb_f1s_exp = [], [], []

for train_idx, test_idx in cv.split(X_expanded, y_expanded):
    Xtr, Xte = X_expanded.iloc[train_idx].values, X_expanded.iloc[test_idx].values
    ytr, yte = y_expanded[train_idx], y_expanded[test_idx]
    
    # Apply SMOTE
    smote = SMOTE(random_state=42, k_neighbors=3)
    Xtr_resampled, ytr_resampled = smote.fit_resample(Xtr, ytr)
    
    # Logistic
    lr_model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    lr_model.fit(Xtr_resampled, ytr_resampled)
    lr_proba = lr_model.predict_proba(Xte)[:, 1]
    lr_pred = (lr_proba >= 0.5).astype(int)
    lr_aucs_exp.append(roc_auc_score(yte, lr_proba))
    lr_f1s_exp.append(f1_score(yte, lr_pred, zero_division=0))
    lr_all_y_true_exp.extend(yte)
    lr_all_y_pred_proba_exp.extend(lr_proba)
    
    # Random Forest
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
    rf_model.fit(Xtr_resampled, ytr_resampled)
    rf_proba = rf_model.predict_proba(Xte)[:, 1]
    rf_pred = (rf_proba >= 0.5).astype(int)
    rf_aucs_exp.append(roc_auc_score(yte, rf_proba))
    rf_f1s_exp.append(f1_score(yte, rf_pred, zero_division=0))
    rf_all_y_true_exp.extend(yte)
    rf_all_y_pred_proba_exp.extend(rf_proba)
    
    # XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=(ytr_resampled == 0).sum() / (ytr_resampled == 1).sum(),
        random_state=42,
        n_jobs=-1,
        eval_metric='auc'
    )
    xgb_model.fit(Xtr_resampled, ytr_resampled, verbose=False)
    xgb_proba = xgb_model.predict_proba(Xte)[:, 1]
    xgb_pred = (xgb_proba >= 0.5).astype(int)
    xgb_aucs_exp.append(roc_auc_score(yte, xgb_proba))
    xgb_f1s_exp.append(f1_score(yte, xgb_pred, zero_division=0))
    xgb_all_y_true_exp.extend(yte)
    xgb_all_y_pred_proba_exp.extend(xgb_proba)

lr_all_y_true_exp = np.array(lr_all_y_true_exp)
lr_all_y_pred_proba_exp = np.array(lr_all_y_pred_proba_exp)
rf_all_y_true_exp = np.array(rf_all_y_true_exp)
rf_all_y_pred_proba_exp = np.array(rf_all_y_pred_proba_exp)
xgb_all_y_true_exp = np.array(xgb_all_y_true_exp)
xgb_all_y_pred_proba_exp = np.array(xgb_all_y_pred_proba_exp)

# Compute lift
lr_lift_exp = compute_lift_metrics(lr_all_y_true_exp, lr_all_y_pred_proba_exp, percentiles=[10, 20, 30])
rf_lift_exp = compute_lift_metrics(rf_all_y_true_exp, rf_all_y_pred_proba_exp, percentiles=[10, 20, 30])
xgb_lift_exp = compute_lift_metrics(xgb_all_y_true_exp, xgb_all_y_pred_proba_exp, percentiles=[10, 20, 30])

# Build comparison table
comparison_expanded = {
    "Logistic Regression": {
        "AUC": np.mean(lr_aucs_exp),
        "F1": np.mean(lr_f1s_exp),
        "Lift@20%": lr_lift_exp["top_20pct"]["lift"],
        "Captured@20%": lr_lift_exp["top_20pct"]["captured_pct"]
    },
    "Random Forest": {
        "AUC": np.mean(rf_aucs_exp),
        "F1": np.mean(rf_f1s_exp),
        "Lift@20%": rf_lift_exp["top_20pct"]["lift"],
        "Captured@20%": rf_lift_exp["top_20pct"]["captured_pct"]
    },
    "XGBoost": {
        "AUC": np.mean(xgb_aucs_exp),
        "F1": np.mean(xgb_f1s_exp),
        "Lift@20%": xgb_lift_exp["top_20pct"]["lift"],
        "Captured@20%": xgb_lift_exp["top_20pct"]["captured_pct"]
    }
}

comparison_expanded_df = pd.DataFrame(comparison_expanded).T

print("\n" + "="*70)
print("PERFORMANCE WITH PAYMENT HISTORY FEATURES")
print("="*70)
print(comparison_expanded_df.round(3))

print("\nIMPROVEMENT OVER BASELINE:")
print("-" * 70)
for model in ["Logistic Regression", "Random Forest", "XGBoost"]:
    auc_old = comparison[model]["AUC"]
    auc_new = comparison_expanded[model]["AUC"]
    lift_old = comparison[model]["Lift@20%"]
    lift_new = comparison_expanded[model]["Lift@20%"]
    print(f"{model:20} | AUC: {auc_old:.3f}→{auc_new:.3f} (+{auc_new-auc_old:+.3f}) | Lift: {lift_old:.3f}→{lift_new:.3f} (+{lift_new-lift_old:+.3f}x)")

best_lift_exp = max(
    comparison_expanded["Logistic Regression"]["Lift@20%"],
    comparison_expanded["Random Forest"]["Lift@20%"],
    comparison_expanded["XGBoost"]["Lift@20%"]
)

print("\n" + "="*70)
if best_lift_exp >= 1.5:
    print(f"✓ MODEL IS NOW VIABLE: Best lift = {best_lift_exp:.2f}x (meets 1.5x threshold!)")
    winner = max(comparison_expanded, key=lambda x: comparison_expanded[x]["Lift@20%"])
    print(f"  → Recommend {winner} for deployment")
else:
    gap = 1.5 - best_lift_exp
    print(f"⚠ STILL BELOW THRESHOLD: Best lift = {best_lift_exp:.3f}x (need +{gap:.3f}x more)")
    print("  → Consider: ensemble stacking, threshold optimization, or additional features")
print("="*70)

In [ ]:
paid = df[df["days_to_payment"].notna()].copy()

bins = [-float("inf"), 30, 60, 90, float("inf")]
labels = ["0-30", "31-60", "61-90", "90+"]
paid["days_bucket"] = pd.cut(paid["days_to_payment"], bins=bins, labels=labels)

bucket_counts = paid["days_bucket"].value_counts().reindex(labels, fill_value=0)

print("=== Payment Timing Summary ===")
print(f"Avg days to pay (mean):   {paid['days_to_payment'].mean():.6f}")
print(f"Avg days to pay (median): {paid['days_to_payment'].median():.6f}")
print()

print("Bucket counts (successful payments only):")
for label in labels:
    print(f"  {label}: {int(bucket_counts[label])}")
print()

print("Bucket rates (all invoices):")
for label in labels:
    print(f"  {label}: {bucket_counts[label] / len(df):.4%}")

not_paid = int(df["days_to_payment"].isna().sum())
print(f"  no_successful_payment_observed: {not_paid} ({not_paid / len(df):.4%})")

## Business Takeaway (Current Cohort)

Most successful payments happen early.

- Paid in 0-30 days: 3966
- Paid in 31-60 days: 18
- Paid in 61-90 days: 3
- Paid after 90 days: 5
- No successful payment observed: 1008

Interpretation:
- If an invoice is unpaid after 30 days, late conversion is possible but rare in this cohort.
- This suggests front-loading outreach/collections in the first 30 days is likely highest leverage.

Caveats:
- This is one cohort and one extraction.
- "No successful payment observed" is not the same as "never paid" outside the observation window.